# E13 · Direct classifier of elevated weeks

Protocol addendum: [`protocolo_revision.md`](../protocolo_revision.md). Written after E1–E8, in response to external feedback; every result here is exploratory. Negative changes favour the first arm; intervals are 95% whole-season bootstrap intervals (20,000 draws, seeds paired).

Formulation S3 of the project: the published count comes from the same Poisson model with the persistence offset; the probability of an elevated week comes from a multi-output XGBoost classifier (outputs at 2, 3 and 4 weeks) with the arm's predictors plus log(k) of each target week, calibrated with chronological Platt. Fixed hyperparameters and seeds as in E1.

In [1]:
import os
import sys
from pathlib import Path

os.environ.update({"DENGUEKAY_XGB_DEVICE": "cpu", "DENGUEKAY_XGB_NJOBS": "2", "OMP_NUM_THREADS": "2",
                   "OPENBLAS_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "MLFLOW_DISABLE_AGENT_HINT": "1"})
STUDY = Path.cwd().parent
sys.path.insert(0, str(STUDY / "code"))

import numpy as np
import pandas as pd

import common as c

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)
import matplotlib.pyplot as plt

import experiments as ex

RES = c.OUTPUT / "resultados"
FIG = STUDY / "figures"
OTHER = tuple(s for s in c.SEASONS if s not in c.DECISION_SEASONS)
read_base = lambda name: pd.read_csv(c.OUTPUT / name, dtype={"ubigeo": str}, parse_dates=["semana_inicio", "origen_cierre"])

import direct_classifier as dc

In [2]:
models = {"classifier, climate": c.read_arm("b3_obs_clasificador", columns=ex.PRED_COLUMNS, folder=dc.OUT_DIR),
          "classifier, no climate": c.read_arm("historia_geo_clasificador", columns=ex.PRED_COLUMNS, folder=dc.OUT_DIR),
          "NB2, climate": c.read_arm("b3_obs", columns=ex.PRED_COLUMNS),
          "NB2, no climate": c.read_arm("historia_geo", columns=ex.PRED_COLUMNS)}
base = read_base("baselines.csv.gz")
KEY = ["semilla", "bloque", "horizonte", "ubigeo", "semana_inicio"]
for arm in ("climate", "no climate"):
    m = models[f"classifier, {arm}"].merge(models[f"NB2, {arm}"], on=KEY, suffixes=("", "_nb2"), validate="one_to_one")
    print(arm, "| rows:", len(m), "| max |count difference|:", float(np.abs(m.mu_modelo - m.mu_modelo_nb2).max()),
          "| same outcomes:", m.brote.equals(m.brote_nb2))

climate | rows: 974025 | max |count difference|: 0.0 | same outcomes: True


no climate | rows: 974025 | max |count difference|: 0.0 | same outcomes: True


In [3]:
s = {k: ex.sse(v) for k, v in models.items()}
s["persistence"] = ex.sse(base, "persistencia_prob")
pairs = [("classifier, climate", "classifier, no climate"), ("classifier, climate", "NB2, climate"),
         ("classifier, no climate", "NB2, no climate"), ("classifier, climate", "persistence"),
         ("classifier, no climate", "persistence"), ("NB2, climate", "NB2, no climate")]
classifier = pd.concat([ex.contrast(s[a], s[b]).assign(comparison=f"{a} vs {b}") for a, b in pairs])
classifier = classifier[["comparison", "horizonte", "brier_a", "brier_b", "cambio_agrupado_%", "ic95_agrupado",
                         "cambio_igual_peso_%", "temporadas_a_mejor"]]
classifier.to_csv(RES / "e13_clasificador.csv", index=False)
classifier.round(4)

,comparison,horizonte,brier_a,brier_b,cambio_agrupado_%,ic95_agrupado,cambio_igual_peso_%,temporadas_a_mejor
0,"classifier, climate vs classifier, no climate",2,0.0286,0.0286,0.3042,"(-1.49, 1.71)",0.1756,8
1,"classifier, climate vs classifier, no climate",3,0.0322,0.0322,-0.1200,"(-2.49, 1.99)",-0.4195,10
2,"classifier, climate vs classifier, no climate",4,0.0356,0.0357,-0.3364,"(-2.94, 2.44)",-0.1329,10
0,"classifier, climate vs NB2, climate",2,0.0286,0.0282,1.4422,"(-2.84, 5.85)",-4.3381,12
1,"classifier, climate vs NB2, climate",3,0.0322,0.0322,0.0806,"(-4.25, 4.01)",-4.1032,12
2,"classifier, climate vs NB2, climate",4,0.0356,0.0356,-0.1657,"(-4.19, 3.57)",-3.3142,11
0,"classifier, no climate vs NB2, no climate",2,0.0286,0.0283,0.8104,"(-2.92, 4.71)",-5.6504,12
1,"classifier, no climate vs NB2, no climate",3,0.0322,0.0323,-0.0757,"(-3.56, 3.19)",-5.1099,13
2,"classifier, no climate vs NB2, no climate",4,0.0357,0.0356,0.1731,"(-3.26, 2.95)",-4.3366,12
0,"classifier, climate vs persistence",2,0.0286,0.0286,-0.0274,"(-5.71, 7.18)",2.6014,11


In [4]:
diag = pd.concat([ex.probability_diagnostics(v[v.horizonte.eq(4)], label=k) for k, v in models.items()])
diag_summary = ex.summarize_diagnostics(diag)
diag_summary.to_csv(RES / "e13_diagnosticos.csv", index=False)
diag_summary.round(4)

,brazo,horizonte,brier,mcb,dsc,unc,ap,pendiente,intercepto
0,"NB2, climate",4,0.0356,0.0044,0.0312,0.0624,0.5031,1.1437,-0.3894
1,"NB2, no climate",4,0.0356,0.0044,0.0312,0.0624,0.5015,1.1300,-0.4549
2,"classifier, climate",4,0.0356,0.0035,0.0303,0.0624,0.5030,1.0720,-0.6612
3,"classifier, no climate",4,0.0357,0.0035,0.0302,0.0624,0.4975,1.1121,-0.4714


## Reading

- The count means equal those of the primary arms (same Poisson model, hyperparameters and seeds), so only the probability route differs.
- Climate effect with the direct classifier: +0.30%, −0.12% and −0.34% at two to four weeks (intervals include zero).
- Classifier vs NB2: +1.44%, +0.08% and −0.17% with climate; +0.81%, −0.08% and +0.17% without (intervals include zero). At four weeks the classifier had lower miscalibration (0.0035 vs 0.0044) and lower discrimination (0.0303 vs 0.0312).
- The absence of a climate effect on probabilities is not an artefact of converting counts to NB2 probabilities.